In [ ]:
%load_ext rich

%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

import json
import mimetypes
import os
import re
import time
from pathlib import Path
from typing import Iterable

import ollama
import requests
from IPython.display import Markdown
from ollama import ChatResponse
from pydantic import BaseModel
from pydantic.json_schema import JsonSchemaValue
from rich.pretty import pprint
from tqdm import tqdm

from aymurai.meta.entities import CanonicalEntity  # , EntityRelation
from aymurai.utils.json_data import get_pretty, load_json, save_json

In [ ]:
os.environ["OLLAMA_KEEP_ALIVE"] = "0"
print("OLLAMA_KEEP_ALIVE set to 0. Models will unload immediately after use.")

In [ ]:
DATA_DIR = "/resources/data/restricted/summarization"
API_URL = "http://localhost:8899"  # Url for debugger. change it to your own

## /document-extract endpoint output

In [ ]:
BASE_URL = os.getenv("DOCUMENT_API_BASE_URL", "http://localhost:8899")
ENDPOINT = f"{BASE_URL}/misc/document-extract"
DATA_ROOT = Path(
    os.getenv("DOCUMENT_DATA_ROOT", "/resources/data/restricted/summarization")
)
DOC_EXTENSIONS = {".pdf", ".docx"}
REQUEST_TIMEOUT_S = float(os.getenv("DOCUMENT_REQUEST_TIMEOUT", "30"))

print(f"Target endpoint: {ENDPOINT}")
print(f"Data root: {DATA_ROOT.resolve()}")

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Directory '{DATA_ROOT}' not found. Update DATA_ROOT before continuing."
    )


def discover_documents(root: Path, extensions: Iterable[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
print(f"Discovered {len(documents)} documents.")

In [ ]:
def call_extraction_api(
    session: requests.Session, file_path: Path
) -> dict[str, object]:
    payload: dict[str, object] = {
        "path": str(file_path),
        "status": "failure",
        "status_code": None,
        "elapsed_s": None,
        "detail": None,
    }

    if not file_path.exists():
        payload["detail"] = "File does not exist"
        return payload

    mime_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    files = {
        "file": (file_path.name, file_path.open("rb"), mime_type),
    }

    try:
        start = time.perf_counter()
        response = session.post(
            ENDPOINT,
            files=files,
            timeout=REQUEST_TIMEOUT_S,
        )
        elapsed = time.perf_counter() - start
    except requests.RequestException as exc:
        payload["detail"] = f"Request failed: {exc}"
        return payload
    finally:
        files["file"][1].close()

    payload["status_code"] = response.status_code
    payload["elapsed_s"] = elapsed

    try:
        response_body = response.json()
    except ValueError:
        response_body = {"raw": response.text[:500]}

    if response.ok:
        payload["status"] = "success"
        payload["detail"] = {
            "document_id": response_body.get("document_id"),
            "document": response_body.get("document", []),
        }
    else:
        payload["detail"] = response_body

    return payload

## Inference

In [ ]:
# Function to make inference using the API
def get_predictions(sample: str) -> dict:
    response = requests.post(url=f"{API_URL}/anonymizer/predict", json={"text": sample})
    response.raise_for_status()
    return response.json()

In [ ]:
from itertools import chain
from operator import itemgetter
from typing import Any

from more_itertools import unique_everseen


def parse_prediction_labels(predictions: list[dict[str, Any]]) -> list[dict[str, str]]:
    """
    Parse prediction labels to extract unique aymurai_label and aymurai_alt_text pairs.

    Args:
        predictions (list[dict[str, Any]]): A list of prediction dictionaries.

    Returns:
        list[dict[str, str]]: A list of dictionaries containing unique aymurai_label and aymurai_alt_text pairs.
    """
    attrs_stream = (
        label.get("attrs") or {}
        for label in chain.from_iterable(pred.get("labels", ()) for pred in predictions)
    )

    unique_pairs = unique_everseen(
        (
            attrs.get("aymurai_label"),
            attrs.get("aymurai_alt_text"),
        )
        for attrs in attrs_stream
        if attrs.get("aymurai_label") and attrs.get("aymurai_alt_text")
    )

    return sorted(
        ({"aymurai_label": label, "text": text} for label, text in unique_pairs),
        key=itemgetter("aymurai_label", "text"),
    )

## CanonicalEntity extraction

In [ ]:
# Function to get chat response from the model
def get_chat_response(
    user_prompt: str,
    model: str = "gpt-oss:20b",
    system_prompt: str = "Sos un asistente jurídico que resume sin inventar nada.",
    options: dict = {"temperature": 0},
    format: JsonSchemaValue | None = None,
) -> ChatResponse:
    """
    Get chat response from the model.

    Args:
        user_prompt (str): The prompt from the user.
        model (str, optional): The model to use. Defaults to "gpt-oss:20b".
        system_prompt (str, optional): The system prompt. Defaults to "Sos un asistente jurídico que resume sin inventar nada.".
        options (dict, optional): The options for the chat. Defaults to {"temperature": 0}.
        format (JsonSchemaValue | None, optional): The format for the response. Defaults to None.

    Returns:
        ChatResponse: The response from the chat.
    """
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        options=options,
        format=format,
    )

    return response

In [ ]:
# Sanity check
get_chat_response("hola, ¿cómo estás?")

In [ ]:
class CanonicalEntities(BaseModel):
    canonical_entities: list[CanonicalEntity]

In [ ]:
system_prompt = """
Eres un asistente especializado en anonimización de sentencias judiciales.
Tu tarea es agrupar las menciones de entidades nombradas detectadas por un modelo de NER en **entidades canónicas** sin omitir ninguna mención válida.
Una **entidad canónica** es la representación única de una entidad real.
Agrupa todas las menciones textuales (aliases) que se refieren a una misma persona, documento, lugar, número, fecha, etc., aunque aparezcan con variantes ortográficas o abreviadas.

# Reglas
- Usa solo información presente en el documento y en las menciones del NER; no inventes datos ni concluyas hechos no expresos.
- Toda mención listada por el NER debe evaluarse. Si representa una entidad real, inclúyela en alguna entidad canónica.
- Puede haber falsos positivos y/o negativos, menciones ambiguas o incompletas, por lo que debes evaluar cada mención cuidadosamente.
- Cada entidad canónica debe incluir:
  - `aymurai_label` (etiqueta del NER, p. ej. "PER", "DNI", etc.).
  - `canonical_text` (forma normalizada: nombres completos, fechas normalizadas, etc.).
  - `aliases` (todas las menciones textuales relevantes).
  - `attributes` (diccionario opcional con roles u otras notas, p. ej. {"role": "Juez/a"}).

# Notas
- `aliases` debe conservar las menciones textuales tal como aparecen en el documento.
- `attributes` es especialmente útil para distinguir roles procesales. Entre ellos pueden incluirse:
    * "Denunciante"
    * "Denunciado/a"
    * "Víctima"
    * "Juez/a"
    * "Fiscal"
    * "Abogado/a"
    * "Perito/a"
    * "Testigo"

# Ejemplo
Q: Con fecha 3 de marzo de 2023, la Sra. Laura Beatriz Gómez, DNI 42.987.654, con domicilio en calle Falsa 123, Barrio Los Pinos, Villa Azul, denunció a su expareja, el Sr. Martín Alberto Rodríguez, DNI 27.654.321, con domicilio en calle Real 789, por hechos de violencia física, psicológica y amenazas con armas blancas.
A: ```json
[
  {
    "aymurai_label": "PER",
    "canonical_text": "Laura Beatriz Gómez",
    "aliases": ["Laura Beatriz Gómez"],
    "attributes": {"role": "Denunciante"}
  },
  {
    "aymurai_label": "DNI",
    "canonical_text": "42987654",
    "aliases": ["42.987.654"]
  },
  {
    "aymurai_label": "DIRECCION",
    "canonical_text": "calle Falsa 123",
    "aliases": ["calle Falsa 123"]
  },
  {
    "aymurai_label": "LOC",
    "canonical_text": "Barrio Los Pinos, Villa Azul",
    "aliases": ["Barrio Los Pinos, Villa Azul"]
  },
  {
    "aymurai_label": "PER",
    "canonical_text": "Martín Alberto Rodríguez",
    "aliases": ["Martín Alberto Rodríguez"],
    "attributes": {"role": "Denunciado"}
  },
  {
    "aymurai_label": "DNI",
    "canonical_text": "27654321",
    "aliases": ["27.654.321"]
  },
  {
    "aymurai_label": "LOC",
    "canonical_text": "calle Real 789",
    "aliases": ["calle Real 789"]
  }
]
```
"""


In [ ]:
user_prompt_template = """
A continuación se proporciona un documento judicial y las menciones de entidades nombradas detectadas por un modelo de NER.

# Documento
{document_text}

# Menciones de entidades detectadas por el NER
{ner_output_json}

# Instrucciones
1. Evalúa cada mención listada. Si representa una entidad real, inclúyela en la lista de aliases de alguna entidad canónica.
2. No omitas entidades válidas, alias con iniciales ni variantes abreviadas.
3. Solo fusiona menciones cuando exista evidencia clara de que se refieren a la misma entidad.
4. Normaliza canonical_text; mantén los alias tal como aparecen.
5. Utiliza attributes para indicar roles (p. ej. {{"role": "Denunciante"}}) o aclaraciones de desambiguación.
"""

In [ ]:
def extract_canonical_entities(
    doc_path: str, model: str = "phi4:14b"
) -> CanonicalEntities:
    """
    Extract canonical entities from a document.

    Args:
        doc_path (str): The path to the document.
        model (str, optional): The model to use for extraction. Defaults to "phi4:14b".

    Raises:
        ValueError: If the document is empty or not found.
        ValueError: If the document summary is empty or not found.

    Returns:
        CanonicalEntities: The extracted canonical entities.
    """
    # Extract document
    session = requests.Session()
    document = call_extraction_api(session, Path(doc_path))
    document = document.get("detail", {}).get("document")

    if not document:
        raise ValueError("Document text is empty or not found.")

    # Get NER predictions
    ner_predictions = [get_predictions(paragraph) for paragraph in tqdm(document)]
    parsed_ner_labels = parse_prediction_labels(ner_predictions)
    ner_output_json = get_pretty(parsed_ner_labels)

    # Prepare user prompt
    user_prompt = user_prompt_template.format(
        document_text="\n".join(document).strip(),
        ner_output_json=ner_output_json,
    )

    # Get chat response
    response = get_chat_response(
        user_prompt=user_prompt,
        model=model,
        system_prompt=system_prompt,
        format=CanonicalEntities.model_json_schema(),
    )

    # Parse canonical entities from response
    canonical_entities = [
        output for output in json.loads(response.message.content)["canonical_entities"]
    ]

    canonical_entities = [
        CanonicalEntity.model_validate(canonical_entity)
        for canonical_entity in canonical_entities
    ]

    return canonical_entities

In [ ]:
for doc_path in documents:
    print(f"Processing document: {doc_path}")

    try:
        canonical_entities = extract_canonical_entities(doc_path)
        print(f"Extracted {len(canonical_entities)} canonical entities.")

        target_filename = re.sub(
            r"\s+", "-", os.path.splitext(os.path.basename(doc_path))[0]
        )
        target_filename = re.sub(r"-{2,}", "-", target_filename)
        save_json(
            [
                canonical_entity.model_dump()
                | {"entity_id": canonical_entity.entity_id.hex}
                for canonical_entity in canonical_entities
            ],
            f"entities-to-review/{target_filename}-canonical-entities.json",
        )

    except Exception as e:
        print(f"Error processing document {doc_path}: {e}")